In [ ]:
!pip install plotly


In [ ]:
import plotly.express as px
import plotly.graph_objects as go


In [1]:
with open("mobility_profile.py", "w") as f:
    f.write('''import random

SENSOR_IDS = ['a', 'j', 'k', 'o', 'h', 'm']  # Simulated rooms

def simulate_movement(length=30):
    base_pattern = ['a', 'j', 'k', 'k', 'o', 'o', 'j', 'a', 'a']
    movement = []

    while len(movement) < length:
        if random.random() < 0.7:
            movement += base_pattern
        else:
            movement += random.choices(SENSOR_IDS, k=3)
    return movement[:length]
''')


In [2]:
with open("lz_trie.py", "w") as f:
    f.write('''class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0

class LZ78Trie:
    def __init__(self):
        self.root = TrieNode()
        self.dictionary = {}  # {phrase: count}
        self.current_phrase = ""

    def insert(self, symbol):
        self.current_phrase += symbol
        if self.current_phrase not in self.dictionary:
            self.dictionary[self.current_phrase] = 1
            self._add_to_trie(self.current_phrase)
            self.current_phrase = ""
        else:
            self.dictionary[self.current_phrase] += 1

    def _add_to_trie(self, phrase):
        node = self.root
        for char in phrase:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.count += 1

    def get_phrases(self):
        return list(self.dictionary.items())
''')


In [3]:
with open("prediction.py", "w") as f:
    f.write('''import math

class Predictor:
    def __init__(self, trie):
        self.phrases = trie.get_phrases()
        self.total = sum(freq for _, freq in self.phrases)
        self.entropy = self._calculate_entropy()

    def _calculate_entropy(self):
        entropy = 0
        for _, freq in self.phrases:
            p = freq / self.total
            entropy -= p * math.log2(p)
        return entropy

    def get_typical_paths(self, epsilon=0.03):
        typical = []
        for phrase, freq in self.phrases:
            p = freq / self.total
            length = len(phrase)
            threshold = 2 ** (-length * self.entropy)
            if abs(p - threshold) <= epsilon:
                typical.append(phrase)
        return typical
''')


In [4]:
with open("resource_manager.py", "w") as f:
    f.write('''class EnergyManager:
    def __init__(self):
        # Approximate power usage in kW per room/device
        self.room_power = {
            'a': 0.2,  # Bedroom
            'j': 0.3,  # Hallway
            'k': 0.5,  # Kitchen
            'o': 0.6,  # Living Room
            'h': 0.4,  # Dining
            'm': 0.25  # Bathroom
        }

    def calculate_savings(self, typical_paths):
        """
        Estimate energy saved by activating only typical paths.
        Each room is assumed to be used for a fixed duration.
        """
        usage_time = 1.0  # Assume 1 hour usage per room
        total = 0
        used_rooms = set("".join(typical_paths))

        for room in used_rooms:
            power = self.room_power.get(room, 0.3)
            total += power * usage_time

        baseline = sum(self.room_power.values()) * usage_time
        savings = baseline - total
        return round(savings, 2)
''')


In [5]:
with open("comfort_model.py", "w") as f:
    f.write('''import math

class ComfortModel:
    def __init__(self):
        self.temp_range = (68, 75)  # Ideal temp in °F
        self.humidity_range = (55, 65)  # Ideal humidity in %
        self.max_temp_comfort = 5
        self.max_humidity_comfort = 5

    def evaluate(self, temp, humidity):
        temp_dev = abs(self._center(self.temp_range) - temp)
        hum_dev = abs(self._center(self.humidity_range) - humidity)

        temp_score = 2 * self.max_temp_comfort / (1 + math.exp(temp_dev))
        hum_score = 2 * self.max_humidity_comfort / (1 + math.exp(hum_dev))

        return round(temp_score * hum_score / 10, 2)

    def _center(self, r):
        return (r[0] + r[1]) / 2
''')


In [12]:


# Enable interactive widgets (for Colab)
from google.colab import output
output.enable_custom_widget_manager()

# Imports
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from mobility_profile import simulate_movement
from lz_trie import LZ78Trie
from prediction import Predictor
from resource_manager import EnergyManager
from comfort_model import ComfortModel

def main():
    print("🏠 SMART HOME VISUAL DASHBOARD\n")

    # Step 1: Simulate Movement
    movements = simulate_movement()
    print("👣 Movement Sequence:", " → ".join(movements[:20]), "..." if len(movements) > 20 else "")

    # 🌐 Room Visit Frequency (Bar Chart)
    room_counts = Counter(movements)
    fig1 = px.bar(x=list(room_counts.keys()), y=list(room_counts.values()),
                  labels={'x': 'Room (Sensor ID)', 'y': 'Visit Count'},
                  title='Room Visit Frequency',
                  color=list(room_counts.keys()))
    fig1.update_layout(template='plotly_white')
    fig1.show()

    # 🔁 Movement Path Timeline (Line Chart)
    steps = list(range(1, len(movements) + 1))
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=steps,
        y=movements,
        mode='lines+markers',
        line=dict(shape='hv', color='mediumvioletred'),
        marker=dict(size=10),
        name='Movement Path'
    ))
    fig2.update_layout(
        title="👣 Inhabitant Movement Path Over Time",
        xaxis_title="Step",
        yaxis_title="Room (Sensor ID)",
        template="plotly_white",
        height=400
    )
    fig2.show()

    # 🔀 Room-to-Room Transitions (Sankey Diagram)
    transitions = [(movements[i], movements[i+1]) for i in range(len(movements)-1)]
    transition_counts = Counter(transitions)
    labels = list(set([room for pair in transition_counts for room in pair]))
    label_index = {label: i for i, label in enumerate(labels)}
    sources = [label_index[a] for (a, b) in transition_counts]
    targets = [label_index[b] for (a, b) in transition_counts]
    values  = list(transition_counts.values())

    fig3 = go.Figure(data=[go.Sankey(
        node=dict(label=labels, pad=15, thickness=20),
        link=dict(source=sources, target=targets, value=values)
    )])
    fig3.update_layout(title_text="🚪 Room-to-Room Movement Flow", font_size=12)
    fig3.show()

    # Step 2: LZ-78 Dictionary
    trie = LZ78Trie()
    for s in movements:
        trie.insert(s)

    phrase_data = list(trie.dictionary.items())[:10]
    if phrase_data:
        phrases, freqs = zip(*phrase_data)
        fig4 = px.bar(x=phrases, y=freqs,
                      labels={'x': 'LZ-78 Phrase', 'y': 'Frequency'},
                      title='Top LZ-78 Phrases',
                      color=phrases)
        fig4.update_layout(template='plotly_white')
        fig4.show()
    else:
        print("❌ No LZ-78 phrases found.")

    # Step 3: Predict Typical Paths
    predictor = Predictor(trie)
    typical_paths = predictor.get_typical_paths()

    if typical_paths:
        fig5 = px.bar(x=typical_paths[:10], y=[1]*len(typical_paths[:10]),
                      labels={'x': 'Typical Path', 'y': 'Probability'},
                      title='Predicted Typical Movement Paths',
                      color=typical_paths[:10])
        fig5.update_layout(template='plotly_white')
        fig5.show()
    else:
        print("❌ No typical paths found.")

    # Step 4: Energy & Comfort
    energy_mgr = EnergyManager()
    comfort_model = ComfortModel()

    saved_energy = energy_mgr.calculate_savings(typical_paths)
    comfort_score = comfort_model.evaluate(temp=72, humidity=60)

    fig6 = go.Figure(data=[go.Pie(
        labels=['Energy Saved (kWh)', 'Comfort Score (/10)'],
        values=[saved_energy, comfort_score],
        hole=0.4,
        marker_colors=['gold', 'lightcoral']
    )])
    fig6.update_layout(title="⚡ Smart Home Benefits Summary", template='plotly_white')
    fig6.show()

    # Final Summary
    print("🎯 Summary")
    print(f"   - Total Rooms Visited: {len(set(movements))}")
    print(f"   - Total LZ-78 Phrases Extracted: {len(trie.dictionary)}")
    print(f"   - Typical Paths Predicted: {len(typical_paths)}")
    print(f"   - ⚡ Energy Saved: {saved_energy:.2f} kWh")
    print(f"   - 😌 Comfort Score: {comfort_score:.2f}/10")

# Run it
main()


🏠 SMART HOME VISUAL DASHBOARD

👣 Movement Sequence: a → j → k → k → o → o → j → a → a → a → j → k → k → o → o → j → a → a → m → h ...


🎯 Summary
   - Total Rooms Visited: 6
   - Total LZ-78 Phrases Extracted: 15
   - Typical Paths Predicted: 6
   - ⚡ Energy Saved: 0.65 kWh
   - 😌 Comfort Score: 1.89/10
